In [4]:
import tensorflow as tf
from tensorflow import keras
from sklearn.model_selection import train_test_split
import numpy as np

In [5]:
(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()

# Нормализуем пиксели изображений в диапазоне [0, 1]
x_train = x_train.astype('float32') / 255.0
x_test = x_test.astype('float32') / 255.0

# Разделяем данные на обучающую и тестовую выборки
x_train, x_val, y_train, y_val = train_test_split(x_train, y_train, test_size=0.2, random_state=42)

In [26]:
# Создаем модель нейронной сети
model = keras.Sequential([
    # Слой свертки с 32 фильтрами и размером ядра 3x3
    keras.layers.Conv2D(32, (3, 3), activation='relu', input_shape=(28, 28, 1)),
    # Слой максимального пулинга с размером пулинга 2x2
    keras.layers.MaxPooling2D((2, 2)),
    # Слой свертки с 64 фильтрами и размером ядра 3x3
    keras.layers.Conv2D(64, (3, 3), activation='relu'),
    # Слой максимального пулинга с размером пулинга 2x2
    keras.layers.MaxPooling2D((2, 2)),
    # Слой плоского преобразования для подготовки данных к полносвязному слою
    keras.layers.Flatten(),
    # Полносвязный слой с 128 нейронами и функцией активации ReLU
    keras.layers.Dense(128, activation='relu'),
    # Выходной слой с 10 нейронами (по одному для каждого класса цифр) и функцией активации softmax
    keras.layers.Dense(10, activation='softmax')
])

In [28]:
# Компилируем модель с функцией потерь categorical_crossentropy и оптимизатором Adam
model.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

In [29]:
model.summary()

Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_6 (Conv2D)               │ (None, 26, 26, 32)     │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_6 (MaxPooling2D)  │ (None, 13, 13, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_7 (Conv2D)               │ (None, 11, 11, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_7 (MaxPooling2D)  │ (None, 5, 5, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_3 (Flatten)             │ (None, 1600)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 128)            │       204,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 10)             │         1,290 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 225,034 (879.04 KB)

 Trainable params: 225,034 (879.04 KB)

 Non-trainable params: 0 (0.00 B)

In [17]:
# Обучаем модель на обучающей выборке
model.fit(
    x_train,
    y_train,
    epochs=10,
    validation_data=(x_val, y_val),
    verbose=False
)

In [18]:
# Оцениваем качество модели на тестовой выборке
test_loss, test_acc = model.evaluate(x_test, y_test)
print(f'Test accuracy: {test_acc:.2f}')

313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9891 - loss: 0.0840
Test accuracy: 0.99


In [23]:
# Используем обученную модель для распознавания чисел
def recognize_number(image):
    # Нормализуем пиксели изображения в диапазоне [0, 1]
    image = image.astype('float32') / 255.0
    # Добавляем размерность批 для изображения
    image = np.expand_dims(image, axis=0)
    # Добавляем размерность канала для изображения (в данном случае 1, поскольку изображение в оттенках серого)
    image = np.expand_dims(image, axis=-1)
    # Получаем предсказание модели
    prediction = model.predict(image)
    # Возвращаем класс с наибольшим значением вероятности
    return np.argmax(prediction)


# Тестируем функцию распознавания чисел
image = x_test[4]
recognized_number = recognize_number(image)
print(f'Распознанное число: {recognized_number}')

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step
Распознанное число: 1
